#### Scrape a website that requires an input

1. Let us try to build it using one plate number

In [ ]:
from playwright.async_api import async_playwright
from playwright.async_api import TimeoutError as PlaywrightTimeoutError

## plate examples
plates = [
    "245QEX",
    "030MLM",
    "949QAD",
    "290MVX",
    "410MDC",
]
## url we are scraping
url="https://medfordor.municipalonlinepayments.com/medfordor/court/search"

## start browser
p = await async_playwright().start()
browser = await p.chromium.launch(headless=False, slow_mo=500)
page = await browser.new_page()
## open this url
await page.goto(url)
## hit that Driver's License button
await page.get_by_role("button", name="Driver's License", exact=True).click()
## locate dropdown and click on it
dropdown = page.locator('#DriversLicenseState-select')
await dropdown.click()

## get all options from dropdown
states = await page.get_by_role("option").all_inner_texts()

## can also use this but we already stored options as list in states
# await dropdown.press("ArrowDown")
# await dropdown.press("Enter")
# await dropdown.press("ArrowUp")

## Navigate to DL Number box using a label
## fill it with one plate
await page.get_by_label("DL Number").fill(plates[0])
## hit that search button
await page.locator(".search-button", has_text="Search").click()

## wait for toast message
toast = page.locator("forge-toast[open]")
await toast.wait_for(timeout=5000)
## grab whatever text pops up
message =await toast.inner_text()
## print print print
print(message)

## Now let us do it properly

In [ ]:
#import libraries
from playwright.async_api import async_playwright
from playwright.async_api import TimeoutError as PlaywrightTimeoutError
import pandas as pd
## plate examples
plates = [
    "245QEX",
    "030MLM",
    "949QAD",
    "290MVX",
    "410MDC",
]

# url 
url = "https://medfordor.municipalonlinepayments.com/medfordor/court/search"

## playwright set up to initiatlize a browser
p = await async_playwright().start()
browser = await p.chromium.launch(headless=False, slow_mo=500)
page = await browser.new_page()
# got to this page
await page.goto(url)
## click on Driver's License button
await page.get_by_role("button", name="Driver's License", exact=True).click()
# select dropdown
dropdown = page.locator("#DriversLicenseState-select")
# Read the list of states once. REVEAL
await dropdown.click()
## stores these in a variable
states = await page.get_by_role("option").all_inner_texts()
await page.keyboard.press("Escape")   # close the list
states = [s.strip() for s in states if s.strip()]   # drop blanks
print(len(states), "states:", states)

## store results in a list
results = []
try:
    # loop through the plates. Taking one plate at a time
    for plate in plates:
        # fill in the plate number
        await page.get_by_label("DL Number").fill(plate)
        # now loop through the states
        for state in states:
            # Choose this state
            await dropdown.click()
            await page.get_by_role("option", name=state, exact=True).click()

            await page.locator(".search-button", has_text="Search").click()
            await page.wait_for_load_state("networkidle")

            toast = page.locator("forge-toast[open]")

            try:
                await toast.wait_for(timeout=5000)
                message = await toast.inner_text()
                message = message.replace("information", "").strip()
                await page.get_by_role("button", name="Dismiss").click()
                await toast.wait_for(state="hidden")   # let it fully close

            except PlaywrightTimeoutError:
                message = "results not found"
                # Going back loads a fresh, empty form, so refill the number
                await page.screenshot(path=f"hit_{plate}_{state}.png")

                await page.goto(url)
                await page.get_by_role("button", name="Driver's License", exact=True).click()
                await page.get_by_label("DL Number").fill(plate)

            print(plate, state, "=", message)
            results.append({"dl_number": plate, "state": state, "message": message})

            await page.wait_for_timeout(1000)

finally:
    # Runs even if the script crashes partway through
    df = pd.DataFrame(results)
    df.to_csv("dl_results.csv", index=False)
    print("Saved", len(df), "rows")
    await browser.close()
    await p.stop()

In [ ]:
## locator examples:

# tag[attr]	has the attribute	forge-toast[open]
# tag[attr="value"]	attribute equals this value exactly	input[name="DriversLicenseNumber"]
# tag[attr*="text"]	attribute contains this text	div[class*="card"]
# tag[attr^="text"]	attribute starts with this text	input[id^="Drivers"]
# tag[attr$="text"]	attribute ends with this text	img[src$=".svg"]
# tag[a][b]	has both attributes	forge-toast[open][dismissible]
# #id	shortcut for [id="..."]	#DriversLicenseNumber
# .class	shortcut for a class name	.search-button
# parent child	child anywhere inside parent	#search_form forge-button